# Continental latitude–elevation binning

Runoff onset statistics per continent aggregated over 1° latitude and 100 m elevation bins
(the N-year median, MAD and the sunny–shaded (CHILI) timing difference; the per-year anomaly
panels; the FCF / CHILI correlation panels). Reads `aggregated_results/<version>/continents/`
written by `pipeline/scripts/reduce_partials.py`. Setup cells first, then any section.

The annotated colorbars (month names, 'lower/higher variability', 'sunny/shaded areas melt first') are drawn in place
via `gsro_analysis.colorbars` — presets over the production repo's `plot_utils` builders (since 2026-09; before that they
were pasted on in slides).

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from gsro_analysis import aggregate, colorbars, paths, settings

In [ ]:
config = settings.load_config()  # the dataset version lives in settings.CONFIG_FILE

PIX_THRESH = 1000   # a (continent, latitude, elevation) bin needs > 1000 pixels to be shown

# the continents cube: continent x latitude x elevation x chili_class x water_year, with
# <var> = bin mean, <var>_n = pixel count (see gsro_analysis.aggregate). No 'statistic' axis.
continents_ds = aggregate.open_aggregate('continents', config.version)

# all insolation classes together (count-weighted), thresholded
ds = aggregate.threshold(aggregate.collapse(continents_ds), PIX_THRESH)

# sunny (warm) minus shaded (cool) CHILI classes, each thresholded on its own count
cool = aggregate.threshold(continents_ds.sel(chili_class='cool'), PIX_THRESH)
warm = aggregate.threshold(continents_ds.sel(chili_class='warm'), PIX_THRESH)
ds['chili_warm_cool_difference'] = warm['runoff_onset_median'] - cool['runoff_onset_median']
ds['chili_warm_cool_ratio'] = warm['runoff_onset_median'] / cool['runoff_onset_median']
ds['chili_warm_cool_n'] = xr.ufuncs.minimum(warm['runoff_onset_median_n'], cool['runoff_onset_median_n'])

# GTOPO30 land-pixel histogram: the grey background of every panel (all land, not just mapped pixels)
dem_pixel_count = continents_ds['dem_pixel_count']
ds

## Share of mapped pixels above 5000 m (unfiltered dataset)

In [ ]:
# the 'full_dataset' filter keeps every seasonal-snow pixel regardless of forest cover
full_ds = aggregate.open_aggregate('continents', config.version, 'full_dataset')
count_data = full_ds['runoff_onset_median_n'].sum('chili_class')
threshold = 5000
total_count = count_data.sum()
above_threshold_count = count_data.sel(elevation=slice(threshold, None)).sum()
print(f"Proportion of pixels above {threshold}m: {float(above_threshold_count / total_count):.2%}")
print(f"Total pixels: {float(total_count):,.0f}")
print(f"Pixels above {threshold}m: {float(above_threshold_count):,.0f}")

## Median, MAD and sunny–shaded difference per continent

In [ ]:
pixel_count_thresh = 20
elev_lat_da = xr.where(dem_pixel_count>pixel_count_thresh, 1, np.nan)

# merge continents north and south america, europe and africa, asia and oceania
na_sa_elev_lat_da = elev_lat_da.sel(continent=['North America','South America']).sum(dim='continent')
na_sa_elev_lat_da = xr.where(na_sa_elev_lat_da>0, 1, np.nan)
eu_af_elev_lat_da = elev_lat_da.sel(continent=['Europe','Africa']).sum(dim='continent')
eu_af_elev_lat_da = xr.where(eu_af_elev_lat_da>0, 1, np.nan)
asia_oceania_elev_lat_da = elev_lat_da.sel(continent=['Asia','Oceania']).sum(dim='continent')
asia_oceania_elev_lat_da = xr.where(asia_oceania_elev_lat_da>0, 1, np.nan)

In [ ]:
# Create figure with 11 subplots (9 plots + 2 spacers)
f, axs = plt.subplots(1, 11, figsize=(21/1.5, 12/1.5), sharey=True, dpi=300,
                      gridspec_kw={'width_ratios': [1, 1, 1, 0.1, 1, 1, 1, 0.1, 1, 1, 1], 'wspace':0.03})

# Define the nice latitude ticks and labels
lat_ticks = [-60, -50, -40, -30, -20, -10, 0, 10, 20, 30, 40, 50, 60, 70, 80]
lat_labels = ['60°S', '50°S', '40°S', '30°S', '20°S', '10°S', '0°', '10°N', '20°N', '30°N', '40°N', '50°N', '60°N', '70°N', '80°N']

# Create a custom colormap for the binary background data
binary_cmap = matplotlib.colors.ListedColormap(['lightgray'])
binary_cmap.set_bad('none', alpha=0)  # Make NaN values completely transparent
binary_cmap_alpha = 1

# ------ Americas plots (0,1,2) ------
# Column 0: Runoff onset median
_ = na_sa_elev_lat_da.plot(ax=axs[0], add_colorbar=False, cmap=binary_cmap, alpha=binary_cmap_alpha, zorder=1)
im1 = ds.sel(continent='North America')['runoff_onset_median'].plot(
    ax=axs[0], add_colorbar=False, cmap='viridis', vmin=100, vmax=300, zorder=2)
_ = ds.sel(continent='South America')['runoff_onset_median'].plot(
    ax=axs[0], add_colorbar=False, cmap='viridis', vmin=100, vmax=300, zorder=2)
axs[0].axhline(y=15, color='black', linestyle='-', linewidth=1)
axs[0].set_title('')

# Column 1: MAD
_ = na_sa_elev_lat_da.plot(ax=axs[1], add_colorbar=False, cmap=binary_cmap, alpha=binary_cmap_alpha, zorder=1)
im2 = ds.sel(continent='North America')['runoff_onset_mad'].plot(
    ax=axs[1], add_colorbar=False, cmap='Reds', vmin=0, vmax=30, zorder=2)
_ = ds.sel(continent='South America')['runoff_onset_mad'].plot(
    ax=axs[1], add_colorbar=False, cmap='Reds', vmin=0, vmax=30, zorder=2)
axs[1].axhline(y=15, color='black', linestyle='-', linewidth=1)
axs[1].set_title('')

# Column 2: CHILI difference
_ = na_sa_elev_lat_da.plot(ax=axs[2], add_colorbar=False, cmap=binary_cmap, alpha=binary_cmap_alpha, zorder=1)
im3 = ds.sel(continent='North America')['chili_warm_cool_difference'].plot(
    ax=axs[2], add_colorbar=False, cmap='PuOr', vmin=-30, vmax=30, zorder=2)
_ = ds.sel(continent='South America')['chili_warm_cool_difference'].plot(
    ax=axs[2], add_colorbar=False, cmap='PuOr', vmin=-30, vmax=30, zorder=2)
axs[2].axhline(y=15, color='black', linestyle='-', linewidth=1)
axs[2].set_title('')

# ------ Europe-Africa plots (4,5,6) ------
# Column 4: Runoff onset median
_ = eu_af_elev_lat_da.plot(ax=axs[4], add_colorbar=False, cmap=binary_cmap, alpha=binary_cmap_alpha, zorder=1)
im4 = ds.sel(continent='Europe')['runoff_onset_median'].plot(
    ax=axs[4], add_colorbar=False, cmap='viridis', vmin=100, vmax=300, zorder=2)
_ = ds.sel(continent='Africa')['runoff_onset_median'].plot(
    ax=axs[4], add_colorbar=False, cmap='viridis', vmin=100, vmax=300, zorder=2)
axs[4].axhline(y=35, color='black', linestyle='-', linewidth=1)
axs[4].set_title('')

# Column 5: MAD
_ = eu_af_elev_lat_da.plot(ax=axs[5], add_colorbar=False, cmap=binary_cmap, alpha=binary_cmap_alpha, zorder=1)
im5 = ds.sel(continent='Europe')['runoff_onset_mad'].plot(
    ax=axs[5], add_colorbar=False, cmap='Reds', vmin=0, vmax=30, zorder=2)
_ = ds.sel(continent='Africa')['runoff_onset_mad'].plot(
    ax=axs[5], add_colorbar=False, cmap='Reds', vmin=0, vmax=30, zorder=2)
axs[5].axhline(y=35, color='black', linestyle='-', linewidth=1)
axs[5].set_title('')

# Column 6: CHILI difference
_ = eu_af_elev_lat_da.plot(ax=axs[6], add_colorbar=False, cmap=binary_cmap, alpha=binary_cmap_alpha, zorder=1)
im6 = ds.sel(continent='Europe')['chili_warm_cool_difference'].plot(
    ax=axs[6], add_colorbar=False, cmap='PuOr', vmin=-30, vmax=30, zorder=2)
_ = ds.sel(continent='Africa')['chili_warm_cool_difference'].plot(
    ax=axs[6], add_colorbar=False, cmap='PuOr', vmin=-30, vmax=30, zorder=2)
axs[6].axhline(y=35, color='black', linestyle='-', linewidth=1)
axs[6].set_title('')

# ------ Asia-Oceania plots (8,9,10) ------
# Column 8: Runoff onset median
_ = asia_oceania_elev_lat_da.plot(ax=axs[8], add_colorbar=False, cmap=binary_cmap, alpha=binary_cmap_alpha, zorder=1)
im8 = ds.sel(continent='Asia')['runoff_onset_median'].plot(
    ax=axs[8], add_colorbar=False, cmap='viridis', vmin=100, vmax=300, zorder=2)
_ = ds.sel(continent='Oceania')['runoff_onset_median'].plot(
    ax=axs[8], add_colorbar=False, cmap='viridis', vmin=100, vmax=300, zorder=2)
axs[8].axhline(y=-12, color='black', linestyle='-', linewidth=1)
axs[8].set_title('')

# Column 9: MAD
_ = asia_oceania_elev_lat_da.plot(ax=axs[9], add_colorbar=False, cmap=binary_cmap, alpha=binary_cmap_alpha, zorder=1)
im9 = ds.sel(continent='Asia')['runoff_onset_mad'].plot(
    ax=axs[9], add_colorbar=False, cmap='Reds', vmin=0, vmax=30, zorder=2)
_ = ds.sel(continent='Oceania')['runoff_onset_mad'].plot(
    ax=axs[9], add_colorbar=False, cmap='Reds', vmin=0, vmax=30, zorder=2)
axs[9].axhline(y=-12, color='black', linestyle='-', linewidth=1)
axs[9].set_title('')

# Column 10: CHILI difference
_ = asia_oceania_elev_lat_da.plot(ax=axs[10], add_colorbar=False, cmap=binary_cmap, alpha=binary_cmap_alpha, zorder=1)
im10 = ds.sel(continent='Asia')['chili_warm_cool_difference'].plot(
    ax=axs[10], add_colorbar=False, cmap='PuOr', vmin=-30, vmax=30, zorder=2)
_ = ds.sel(continent='Oceania')['chili_warm_cool_difference'].plot(
    ax=axs[10], add_colorbar=False, cmap='PuOr', vmin=-30, vmax=30, zorder=2)
axs[10].axhline(y=-12, color='black', linestyle='-', linewidth=1)
axs[10].set_title('')

# Remove the spacer axes
axs[3].remove()
axs[7].remove()

# Common settings for all axes
for ax in axs:
    if ax not in [axs[3], axs[7]]:  # Skip the removed spacer axes
        ax.set_xlim([0, 9000])
        ax.set_ylim([-60, 70])
        ax.set_xlabel('')
        ax.set_ylabel('')
        
        # Set the nice latitude ticks and labels
        ax.set_yticks(lat_ticks)
        if ax == axs[0]:  # First plot in each group
            ax.set_yticklabels(lat_labels)
        else:
            ax.set_yticklabels([])
            ax.tick_params(axis='y', which='both', length=0)  # Hide y-ticks for non-first plots

        # Set elevation ticks
        elev_major_ticks = [0, 2000, 4000, 6000, 8000]
        elev_minor_ticks = [1000, 3000, 5000, 7000, 9000]
        ax.xaxis.set_major_locator(matplotlib.ticker.FixedLocator(elev_major_ticks))
        ax.xaxis.set_minor_locator(matplotlib.ticker.FixedLocator(elev_minor_ticks))

        ax.grid(True, which='major', linestyle='--', alpha=0.5)
        ax.grid(True, which='minor', linestyle='--', alpha=0.5)

        ax.set_xticks(elev_major_ticks)
        ax.set_xticklabels([f"{tick}" for tick in elev_major_ticks], rotation=90, ha='center')
        ax.tick_params(axis='x', which='minor', length=0)  # Set minor ticks length to 0 for x-axis

axs[0].set_yticklabels(lat_labels)  # Ensure labels are set for leftmost plot

plt.subplots_adjust(bottom=0.15)  # Make room for colorbars

# Get the positions of the first and third plot in each group to determine colorbar spans
def get_colorbar_position(first_ax, third_ax, shrink_factor=0.8):
    first_pos = first_ax.get_position()
    third_pos = third_ax.get_position()
    
    # Calculate full width
    full_width = third_pos.x1 - first_pos.x0
    
    # Calculate shrunk width (80% of full width)
    width = full_width * shrink_factor
    
    # Calculate left position (center the colorbar)
    left = first_pos.x0 + (full_width - width) / 2
    
    # Return [left, bottom, width, height]; tall enough for the two rows of month names
    return [left, 0.0, width, 0.035]

# Annotated colorbars, drawn in place (gsro_analysis.colorbars on top of the production
# plot_utils builders): month names on the onset bar, 'lower/higher variability' on the MAD bar,
# 'sunny/shaded areas melt first' on the CHILI bar. Before 2026-09 these were pasted on in slides.
n_years = len(continents_ds.water_year)
cbar_ax1 = f.add_axes(get_colorbar_position(axs[0], axs[2], shrink_factor=0.95))
cbar_ax2 = f.add_axes(get_colorbar_position(axs[4], axs[6], shrink_factor=0.95))
cbar_ax3 = f.add_axes(get_colorbar_position(axs[8], axs[10], shrink_factor=0.95))
colorbars.median_onset(cbar_ax1, n_years, **colorbars.EMBEDDED_MONTH)
colorbars.mad(cbar_ax2, n_years, **colorbars.EMBEDDED)
colorbars.sunny_shaded(cbar_ax3, **colorbars.EMBEDDED)

# Add continent labels on rightmost plots of each set, in bold
# For Americas
axs[2].text(8700, 16, 'North\nAmerica', va='bottom', ha='right', fontweight='bold', fontsize=15)
axs[2].text(8700, 13, 'South\nAmerica', va='top', ha='right', fontweight='bold', fontsize=15)

# For Europe-Africa
axs[6].text(8700, 36, 'Europe', va='bottom', ha='right', fontweight='bold', fontsize=15)
axs[6].text(8700, 33, 'Africa', va='top', ha='right', fontweight='bold', fontsize=15)

# For Asia-Oceania
axs[10].text(8700, -10, 'Asia', va='bottom', ha='right', fontweight='bold', fontsize=15)
axs[10].text(8700, -14, 'Oceania', va='top', ha='right', fontweight='bold', fontsize=15)

# Add shared xlabel
f.text(0.5, 0.065, 'Elevation [meters]', ha='center', fontsize=14)

# Add shared ylabel
f.text(0.075, 0.5, 'Latitude [degrees]', va='center', rotation='vertical', fontsize=14)

# Save the figure
f.savefig(paths.figdir('global', config.version) / 'continent_lat_and_elev_binned.png', dpi=300, bbox_inches='tight')

## Global anomaly panels (median + one column per water year)

In [ ]:
n_cols = 1 + len(ds.water_year)   # the median column + one per water year
f,axs=plt.subplots(3,n_cols,figsize=(20*n_cols/11,15),sharex=True,sharey=True, dpi=300)

lat_ticks = [-60, -50, -40, -30, -20, -10, 0, 10, 20, 30, 40, 50, 60, 70]
lat_labels = ['60°S', '50°S', '40°S', '30°S', '20°S', '10°S', '0°', '10°N', '20°N', '30°N', '40°N', '50°N', '60°N', '70°N']
elev_ticks = [0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000]



ds['runoff_onset_median'].sel(continent='North America').plot(ax=axs[0,0],vmin=100,vmax=300,cmap='viridis',add_colorbar=False)
ds['runoff_onset_median'].sel(continent='South America').plot(ax=axs[0,0],vmin=100,vmax=300,cmap='viridis',add_colorbar=False)

ds['runoff_onset_median'].sel(continent='Europe').plot(ax=axs[1,0],vmin=100,vmax=300,cmap='viridis',add_colorbar=False)
ds['runoff_onset_median'].sel(continent='Africa').plot(ax=axs[1,0],vmin=100,vmax=300,cmap='viridis',add_colorbar=False)

ds['runoff_onset_median'].sel(continent='Asia').plot(ax=axs[2,0],vmin=100,vmax=300,cmap='viridis',add_colorbar=False)
ds['runoff_onset_median'].sel(continent='Oceania').plot(ax=axs[2,0],vmin=100,vmax=300,cmap='viridis',add_colorbar=False)

for i,year in enumerate(ds.water_year.values):
    ds['runoff_onset_anomaly'].sel(continent='North America',water_year=year).plot(ax=axs[0,i+1],vmin=-30,vmax=30,cmap='RdBu',add_colorbar=False)
    ds['runoff_onset_anomaly'].sel(continent='South America',water_year=year).plot(ax=axs[0,i+1],vmin=-30,vmax=30,cmap='RdBu',add_colorbar=False)

    ds['runoff_onset_anomaly'].sel(continent='Europe',water_year=year).plot(ax=axs[1,i+1],vmin=-30,vmax=30,cmap='RdBu',add_colorbar=False)
    ds['runoff_onset_anomaly'].sel(continent='Africa',water_year=year).plot(ax=axs[1,i+1],vmin=-30,vmax=30,cmap='RdBu',add_colorbar=False)

    ds['runoff_onset_anomaly'].sel(continent='Asia',water_year=year).plot(ax=axs[2,i+1],vmin=-30,vmax=30,cmap='RdBu',add_colorbar=False)
    ds['runoff_onset_anomaly'].sel(continent='Oceania',water_year=year).plot(ax=axs[2,i+1],vmin=-30,vmax=30,cmap='RdBu',add_colorbar=False)

for i,ax_row in enumerate(axs):
    for j,ax in enumerate(ax_row):
        ax.set_xlim([0, 8000])
        ax.set_ylim([-60, 70])  # Full latitude range
        ax.grid(True, linestyle='--', alpha=0.5)
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_title('')
        ax.set_yticks(lat_ticks)

        ax.set_xticks(elev_ticks)
        # ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

        if j == 0:
            ax.set_yticklabels(lat_labels)

        if i == 0:
            ax.axhline(y=15, color='black', linestyle='-', linewidth=1)  # Dividing line
            ax.text(7800, 16, 'North\nAmerica', va='bottom', ha='right', fontweight='bold', fontsize=10)
            ax.text(7800, 13, 'South\nAmerica', va='top', ha='right', fontweight='bold', fontsize=10)

            if j == 0:
                ax.set_title('10-year median\nrunoff onset')
                ax.set_ylabel('North America and South America')
            else:
                ax.set_title(f'WY {ds.water_year.values[j-1]}\nanomaly')

        if i == 1:
            ax.axhline(y=35, color='black', linestyle='-', linewidth=1)  # Dividing line
            ax.text(7800, 36, 'Europe', va='bottom', ha='right', fontweight='bold', fontsize=10)
            ax.text(7800, 33, 'Africa', va='top', ha='right', fontweight='bold', fontsize=10)
            if j == 0:
                ax.set_ylabel('Europe and Africa')

        if i == 2:
            ax.axhline(y=-12, color='black', linestyle='-', linewidth=1)  # Dividing line
            ax.text(7800, -10, 'Asia', va='bottom', ha='right', fontweight='bold', fontsize=10)
            ax.text(7800, -14, 'Oceania', va='top', ha='right', fontweight='bold', fontsize=10)
            ax.set_xticks(elev_ticks)
            ax.set_xticklabels(ax.get_xticklabels(), rotation=90, ha='center')
            ax.set_xlabel('elevation [meters]')
            if j == 0:
                ax.set_ylabel('Asia and Oceania')

        


# # For Europe-Africa
# axs[6].text(7800, 36, 'Europe', va='bottom', ha='right', fontweight='bold', fontsize=13)
# axs[6].text(7800, 33, 'Africa', va='top', ha='right', fontweight='bold', fontsize=13)

# # For Asia-Oceania
# axs[10].text(7800, -10, 'Asia', va='bottom', ha='right', fontweight='bold', fontsize=13)
# axs[10].text(7800, -14, 'Oceania', va='top', ha='right', fontweight='bold', fontsize=13)

f.tight_layout()

f.savefig(paths.figdir('global', config.version) / 'continent_lat_and_elev_binned_anomaly.png', dpi=300)

## Data exploration

In [ ]:
continent_order = ['North America', 'Europe', 'Asia', 'South America', 'Africa', 'Oceania']
ds = ds.reindex({'continent': continent_order})
ds

In [ ]:
f,ax=plt.subplots(figsize=(20,10))
ds.sel(continent='North America').plot.scatter(ax=ax,x='runoff_onset_median', y='runoff_onset_mad',hue='elevation')

f,ax=plt.subplots(figsize=(20,10))
ds.sel(continent='North America').plot.scatter(ax=ax,x='runoff_onset_median', y='chili_warm_cool_difference',hue='elevation')

f,ax=plt.subplots(figsize=(10,10))
ds.plot.scatter(ax=ax,x='runoff_onset_median', y='runoff_onset_mad',hue='elevation')

In [ ]:
f,axs=plt.subplots(1,2,figsize=(20,10))
ds.plot.scatter(ax=axs[0],x='runoff_onset_mad', y='chili_warm_cool_difference',hue='latitude')
ds.plot.scatter(ax=axs[1],x='runoff_onset_mad', y='chili_warm_cool_difference',hue='elevation')

In [ ]:
ds.plot.scatter(col='continent',col_wrap=3,x='runoff_onset_mad', y='chili_warm_cool_difference',hue='elevation')

## Forest cover fraction and CHILI correlations with median onset (per bin, pixel-level Pearson r)

**important note to self** 
we should write a bit more about the interpretation of this in the context of lundquist 2013. we should do a similar analysis at the mountain range scale..... 


In [ ]:
f, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(7, 12), sharey=True, dpi=300)

# Define the nice latitude ticks and labels
lat_ticks = [-60, -50, -40, -30, -20, -10, 0, 10, 20, 30, 40, 50, 60, 70]
lat_labels = ['60°S', '50°S', '40°S', '30°S', '20°S', '10°S', '0°', '10°N', '20°N', '30°N', '40°N', '50°N', '60°N', '70°N']

# Plot North America and South America
ds.sel(continent='North America')['fcf_corr'].plot(
    ax=ax1, add_colorbar=False, cmap='RdBu', vmin=-1, vmax=1)
ds.sel(continent='South America')['fcf_corr'].plot(
    ax=ax1, add_colorbar=False, cmap='RdBu', vmin=-1, vmax=1)
ax1.axhline(y=15, color='black', linestyle='-', linewidth=1)  # Dividing line
ax1.set_title('North & South America')

# Plot Europe and Africa
ds.sel(continent='Europe')['fcf_corr'].plot(
    ax=ax2, add_colorbar=False, cmap='RdBu', vmin=-1, vmax=1)
ds.sel(continent='Africa')['fcf_corr'].plot(
    ax=ax2, add_colorbar=False, cmap='RdBu', vmin=-1, vmax=1)
ax2.axhline(y=35, color='black', linestyle='-', linewidth=1)  # Dividing line
ax2.set_title('Europe & Africa')

# Plot Asia and Oceania
ds.sel(continent='Asia')['fcf_corr'].plot(
    ax=ax3, add_colorbar=False, cmap='RdBu', vmin=-1, vmax=1)
ds.sel(continent='Oceania')['fcf_corr'].plot(
    ax=ax3, add_colorbar=False, cmap='RdBu', vmin=-1, vmax=1)
ax3.axhline(y=-12, color='black', linestyle='-', linewidth=1)  # Dividing line
ax3.set_title('Asia & Oceania')

# Common settings for all axes
for ax in [ax1, ax2, ax3]:
    ax.set_xlim([0, 8000])
    ax.set_ylim([-60, 70])  # Full latitude range
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xlabel('elevation [meters]')
    ax.set_ylabel('')
    
    # Set the nice latitude ticks and labels
    ax.set_yticks(lat_ticks)
    if ax == ax1:  # Only set labels for leftmost plot
        ax.set_yticklabels(lat_labels)
    # else:
    #     ax.set_yticklabels([])

# Only leftmost plot needs y-label
#ax1.set_ylabel('latitude [degrees]')

# Add elevation ticks
elev_ticks = [0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000]
for ax in [ax1, ax2, ax3]:
    ax.set_xticks(elev_ticks)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

f.suptitle('FCF correlation with runoff onset\n(red means snow melts earlier with greater fcf)\n(blue means snow melts later with greater fcf)', fontsize=16)

f.tight_layout()

In [ ]:
f, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(7, 12), sharey=True, dpi=300)

# Define the nice latitude ticks and labels
lat_ticks = [-60, -50, -40, -30, -20, -10, 0, 10, 20, 30, 40, 50, 60, 70]
lat_labels = ['60°S', '50°S', '40°S', '30°S', '20°S', '10°S', '0°', '10°N', '20°N', '30°N', '40°N', '50°N', '60°N', '70°N']

# Plot North America and South America
ds.sel(continent='North America')['chili_corr'].plot(
    ax=ax1, add_colorbar=False, cmap='RdBu', vmin=-1, vmax=1)
ds.sel(continent='South America')['chili_corr'].plot(
    ax=ax1, add_colorbar=False, cmap='RdBu', vmin=-1, vmax=1)
ax1.axhline(y=15, color='black', linestyle='-', linewidth=1)  # Dividing line
ax1.set_title('North & South America')

# Plot Europe and Africa
ds.sel(continent='Europe')['chili_corr'].plot(
    ax=ax2, add_colorbar=False, cmap='RdBu', vmin=-1, vmax=1)
ds.sel(continent='Africa')['chili_corr'].plot(
    ax=ax2, add_colorbar=False, cmap='RdBu', vmin=-1, vmax=1)
ax2.axhline(y=35, color='black', linestyle='-', linewidth=1)  # Dividing line
ax2.set_title('Europe & Africa')

# Plot Asia and Oceania
ds.sel(continent='Asia')['chili_corr'].plot(
    ax=ax3, add_colorbar=False, cmap='RdBu', vmin=-1, vmax=1)
ds.sel(continent='Oceania')['chili_corr'].plot(
    ax=ax3, add_colorbar=False, cmap='RdBu', vmin=-1, vmax=1)
ax3.axhline(y=-12, color='black', linestyle='-', linewidth=1)  # Dividing line
ax3.set_title('Asia & Oceania')

# Common settings for all axes
for ax in [ax1, ax2, ax3]:
    ax.set_xlim([0, 8000])
    ax.set_ylim([-60, 70])  # Full latitude range
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_xlabel('elevation [meters]')
    ax.set_ylabel('')
    
    # Set the nice latitude ticks and labels
    ax.set_yticks(lat_ticks)
    if ax == ax1:  # Only set labels for leftmost plot
        ax.set_yticklabels(lat_labels)
    # else:
    #     ax.set_yticklabels([])

# Only leftmost plot needs y-label
#ax1.set_ylabel('latitude [degrees]')

# Add elevation ticks
elev_ticks = [0, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000]
for ax in [ax1, ax2, ax3]:
    ax.set_xticks(elev_ticks)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

f.suptitle('CHILI correlation with runoff onset\n(red means sun-facing slopes melt first)', fontsize=16)

f.tight_layout()